In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
silver_orders = spark.table("workspace.final_project.silver_orders")
silver_items = spark.table("workspace.final_project.silver_items")
silver_suppliers = spark.table("workspace.final_project.silver_suppliers")
silver_incidents = spark.table("workspace.final_project.silver_incidents")

In [0]:
dim_suppliers = (
    silver_suppliers
    .select(
        "supplier_id",
        "supplier_name",
        "city",
        "country",
        "country_code"
    )
)

dim_suppliers.write.mode("overwrite").saveAsTable(
    "workspace.final_project.gold_dim_suppliers"
)

In [0]:
# a preparer peut etre dès la couche silver
fact_order_items = (
    silver_items
    .join(silver_orders, "order_id")
    .withColumn("order_month", F.trunc("order_date", "month"))
    .withColumn("spend", F.col("quantity") * F.col("unit_price"))
)

fact_order_items.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.final_project.gold_fact_order_items")

In [0]:
fact_incidents = (
    silver_incidents
    .withColumn(
        "incident_month",
        F.trunc("incident_date", "month")
    )
)

fact_incidents.write.mode("overwrite").saveAsTable(
    "workspace.final_project.gold_fact_incidents"
)

In [0]:
orders_monthly = (
    fact_order_items
    .groupBy(
        "supplier_id",
        F.year("order_date").alias("year"),
        F.month("order_date").alias("month")
    )
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("quantity").alias("total_quantity"),
        F.sum("spend").alias("total_spend"),
        F.avg("spend").alias("avg_order_value"),

        F.avg(
            F.when(
                F.col("delivery_date_actual") <= F.col("delivery_date_expected"),
                1
            ).otherwise(0)
        ).alias("on_time_delivery_rate"),

        F.avg(
            F.datediff(
                "delivery_date_actual",
                "delivery_date_expected"
            )
        ).alias("avg_delay_days")
    )
)

In [0]:
incidents_with_supplier = (
    silver_incidents
    .join(
        silver_orders.select("order_id", "supplier_id", "order_date"),
        "order_id",
        "left"
    )
)
incidents_monthly = (
    incidents_with_supplier
    .withColumn("year", F.year("order_date"))
    .withColumn("month", F.month("order_date"))
    .groupBy("supplier_id", "year", "month")
    .agg(F.count("*").alias("incident_count"))
)

In [0]:
supplier_monthly = (
    orders_monthly
    .join(
        incidents_monthly,
        ["supplier_id", "year", "month"],
        "left"
    )
    .fillna({"incident_count": 0})
)

supplier_monthly = supplier_monthly.withColumn(
    "incident_rate",
    F.col("incident_count") / F.col("total_orders")
)

In [0]:
supplier_monthly = (
    orders_monthly
    .join(
        incidents_monthly,
        ["supplier_id", "year", "month"],
        "left"
    )
    .fillna({"incident_count": 0})
)

supplier_monthly = supplier_monthly.withColumn(
    "incident_rate",
    F.col("incident_count") / F.col("total_orders")
)

# Exclure le supplier 47
supplier_monthly_filtered = supplier_monthly.filter(
    F.col("supplier_id") != 47
)

In [0]:
display(supplier_monthly_filtered)

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
window_3m = (
    Window
    .partitionBy("supplier_id")
    .orderBy("year", "month")
    .rowsBetween(-2, 0)
)

supplier_monthly = (
    supplier_monthly
    .withColumn(
        "rolling_orders_3m",
        F.sum("total_orders").over(window_3m)
    )
    .withColumn(
        "rolling_spend_3m",
        F.sum("total_spend").over(window_3m)
    )
    .withColumn(
        "rolling_incident_rate_3m",
        F.avg("incident_rate").over(window_3m)
    )
    .withColumn(
        "rolling_delivery_rate_3m",
        F.avg("on_time_delivery_rate").over(window_3m)
    )
)

In [0]:
supplier_monthly.write.mode("overwrite").saveAsTable(
    "workspace.final_project.gold_supplier_performance_monthly"
)